# 04｜第 2 節：YOLO11 現成模型、上傳照片與自訂類別微調

**學習方式：所有 Code Cell 已填好，照順序執行；只需上傳檔案及修改大寫參數。**

這份課程有兩段：**先使用 `yolo11n.pt` 原始模型辨識自己上傳的照片**；再使用本課公開資料集提供的 cat／dog 兩類與方框標註做微調，回到 Colab **微調後再次上傳照片**。請在 Colab 選 GPU（例如 T4），並確保有網路下載模型權重。

標註步驟：先使用 `04_貓狗YOLO資料集.zip`（由 ChatGPT Work 準備；已含標註）→ 在 Colab 上傳；有需要再用 `04_標註工具.html` 匯入與修正方框。該 ZIP 含 `images/`、`labels/`、`data.yaml`，**直接上傳即可，不需手動編輯標註文字**。

注意：原始 YOLO11n 已學會 COCO 的 `cat`、`dog`；本課微調是讓模型適應新的貓狗照片，並不保證成效一定提高。資料集有 Train 貓 100／狗 100，Val 貓 30／狗 30。

## 1. 安裝並載入預訓練 YOLO

In [ ]:
# Install Ultralytics for YOLO11 inference and training.
# 安裝 YOLO11 推論與微調套件。
%pip install -q 'ultralytics>=8.3,<9'

# Import YOLO from Ultralytics.
# 匯入 YOLO 模型。
from ultralytics import YOLO

# Import OpenCV to show prediction images correctly.
# 匯入 OpenCV 轉換預測結果的顏色。
import cv2

# Import image display tools.
# 匯入圖片顯示工具。
from PIL import Image

# Import the Notebook display helper.
# 匯入 Notebook 的圖片顯示工具。
from IPython.display import display

# Import the Colab upload interface.
# 匯入 Colab 的檔案上傳功能。
from google.colab import files

# Import helpers for handling the annotated dataset ZIP.
# 匯入解壓縮標註資料集需要的工具。
from pathlib import Path

# Import ZIP extraction helpers.
# 匯入 ZIP 壓縮檔解壓工具。
from zipfile import ZipFile

# Import helpers for replacing the previous practice folder.
# 匯入清除舊練習資料夾的工具。
import shutil

# Import PyTorch to use a GPU when available.
# 匯入 PyTorch，以便判斷是否有 GPU。
import torch

# Choose the original pretrained detection model.
# 選擇原始預訓練的 YOLO11 Nano 模型。
PRETRAINED_MODEL = 'yolo11n.pt'

# Load the pretrained model; download weights if needed.
# 載入預訓練模型，第一次執行會下載權重。
model = YOLO(PRETRAINED_MODEL)


## 2. 上傳照片，用原始模型畫出偵測框

執行後會出現上傳按鈕。上傳 JPG／PNG；`CONF` 可由 `0.25` 調高到 `0.5`，比較框的數量。**原始模型已包含 `cat`、`dog`，適合先比較微調前的結果。**

In [ ]:
# Set the minimum detection confidence.
# 設定物件偵測的最低信心分數。
CONF = 0.25

# Upload a photograph from your computer.
# 從自己的電腦上傳一張照片。
uploaded = files.upload()

# Use the uploaded image as the input.
# 將上傳的照片作為模型輸入。
IMAGE_PATH = next(iter(uploaded))

# Predict objects with the original YOLO model.
# 使用原始 YOLO 模型偵測物件。
result = model.predict(IMAGE_PATH, conf=CONF, verbose=False)[0]

# Display the real model output with bounding boxes.
# 顯示模型實際預測的方框與類別。
display(Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)))


## 3. 上傳標註工具匯出的資料集 ZIP

執行後上傳教材所附的 `04_貓狗YOLO資料集.zip`（或自行修改後從標註工具匯出的 ZIP），程式會解壓縮到獨立資料夾，不和原始練習照片混在一起。每次重跑會重新建立這個獨立資料夾。本課的類別為 `cat=0`、`dog=1`，Train 200 張、Val 60 張；每張照片都應有對應的方框標註。

`data.yaml` 的類別名稱直接取自標註工具，不需要在 Notebook 手動填寫類別清單。

In [ ]:
# Upload the dataset ZIP exported by the annotation tool.
# 上傳標註工具匯出的完整資料集 ZIP。
label_upload = files.upload()

# Get the uploaded ZIP file.
# 取得上傳的資料集壓縮檔。
ZIP_PATH = next(iter(label_upload))

# Keep the custom labels separate from sample images.
# 將學生標註的資料放在獨立資料夾。
DATA_DIR = Path('04_YOLO學生標註資料').resolve()

# Replace only the previously imported practice dataset.
# 只清除先前匯入的學生標註資料夾。
shutil.rmtree(DATA_DIR, ignore_errors=True)

# Extract the images, labels, and configuration together.
# 一次解壓縮圖片、標註與設定檔。
with ZipFile(ZIP_PATH) as archive:
    # Extract the contents to the practice folder.
    # 將完整資料集解壓到指定資料夾。
    archive.extractall(DATA_DIR)

# Locate the class and dataset configuration.
# 找到標註工具匯出的資料集設定。
data_path = DATA_DIR / 'data.yaml'

# Add the absolute dataset path for reliable training.
# 補上資料集的絕對路徑，確保 YOLO 找得到圖片。
data_path.write_text(f'path: {DATA_DIR}\n' + data_path.read_text(encoding='utf-8'), encoding='utf-8')

# Show the dataset configuration and custom class names.
# 查看資料夾路徑、Train／Val 與自訂類別。
print(data_path.read_text(encoding='utf-8'))


## 4. 微調 YOLO：只改三個參數

預設 `EPOCHS=5`、`IMAGE_SIZE=416`、`BATCH_SIZE=4`，可先確認貓狗微調流程能正常執行。若時間、圖片或 GPU 記憶體有限，可以降低 `BATCH_SIZE`。訓練結果儲存在 `04_訓練成果/student/`；這裡**不會事先假設模型準確度是多少**。希望進一步觀察變化，可提高 Epoch 到 20 或更多，但需看 Validation 指標。

In [ ]:
# Choose how many training epochs to run.
# 設定要訓練幾個 Epoch。
EPOCHS = 5

# Choose the image size used in training.
# 設定訓練時的輸入圖片尺寸。
IMAGE_SIZE = 416

# Choose the training batch size.
# 設定每個 Batch 的圖片數量。
BATCH_SIZE = 4

# Use GPU when available; otherwise use CPU.
# 有 GPU 就使用 GPU，否則使用 CPU。
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# Start from the original pretrained model weights.
# 使用原始預訓練權重開始微調。
finetune_model = YOLO(PRETRAINED_MODEL)

# Fine-tune using the student-created dataset.
# 使用自己框選並匯出的標註資料微調。
finetune_model.train(data=str(data_path), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE, device=DEVICE, project='04_訓練成果', name='student', exist_ok=True)

# Save the path of the best model checkpoint.
# 記錄微調完成後最佳權重的路徑。
BEST_WEIGHTS = str(finetune_model.trainer.best)

# Show where the trained model was saved.
# 顯示最佳模型權重的位置。
print('微調後模型：', BEST_WEIGHTS)


## 5. 再上傳一張照片，比較原始模型與微調模型

上傳你想測試的新照片，程式會**真正載入訓練得到的 `best.pt`** 進行預測。畫面會分別顯示原始模型與微調模型的結果；兩者類別定義可能不同，**不能只比較框數就斷定誰較準**。想測試別張照片，重新執行本格即可。

如果重新開啟 Colab 導致訓練結果不見，需要重新訓練，或先將 `best.pt` 另存並修改 `BEST_WEIGHTS`。

In [ ]:
# Select the confidence threshold for both models.
# 設定原始模型與微調模型共同使用的信心門檻。
TEST_CONF = 0.25

# Upload a new picture for comparison.
# 上傳要比較預測結果的新照片。
test_upload = files.upload()

# Get the new photo filename.
# 取得這張新照片的檔名。
TEST_IMAGE = next(iter(test_upload))

# Load the model produced by fine-tuning.
# 載入剛才微調完成的最佳模型。
custom_model = YOLO(BEST_WEIGHTS)

# Run the original model on this same photo.
# 使用原始模型預測同一張照片。
original_result = model.predict(TEST_IMAGE, conf=TEST_CONF, verbose=False)[0]

# Run the fine-tuned model on the same photo.
# 使用微調模型預測同一張照片。
custom_result = custom_model.predict(TEST_IMAGE, conf=TEST_CONF, verbose=False)[0]

# Show the original model's bounding boxes.
# 顯示原始模型實際預測的結果。
print('原始模型：')

# Show bounding boxes predicted by the original weights.
# 顯示原始權重預測的實際物件方框。
display(Image.fromarray(cv2.cvtColor(original_result.plot(), cv2.COLOR_BGR2RGB)))

# Show the fine-tuned model's bounding boxes.
# 顯示微調模型實際預測的結果。
print('微調模型：')

# Show bounding boxes predicted by the custom weights.
# 顯示自訂權重預測的實際物件方框。
display(Image.fromarray(cv2.cvtColor(custom_result.plot(), cv2.COLOR_BGR2RGB)))


**本節重點：**原始模型適用於原本訓練過的類別；貓與狗需要一致的類別定義、正確的圖片框選，以及足夠多樣的 Train／Val。`04_標註工具.html` 可用來補框與續標。微調只代表模型繼續學習，並不保證一定比原始模型好；需要用**未參與訓練**的照片實際驗證。